Mount Drive and load the raw files

In [8]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np

folder = Path("/content/drive/MyDrive/Project2_SolarPV")

gen_path = folder / "Plant_1_Generation_Data.csv"
wth_path = folder / "Plant_1_Weather_Sensor_Data.csv"

if not gen_path.is_file() or not wth_path.is_file():
    raise FileNotFoundError("Check that both Plant 1 CSV files are in the folder.")

gen = pd.read_csv(gen_path)
wth = pd.read_csv(wth_path)

gen["DATE_TIME"] = pd.to_datetime(
    gen["DATE_TIME"], format="%d-%m-%Y %H:%M", errors="raise"
)
wth["DATE_TIME"] = pd.to_datetime(
    wth["DATE_TIME"], format="%Y-%m-%d %H:%M:%S", errors="raise"
)

print("Generation shape:", gen.shape)
print("Weather shape:", wth.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Generation shape: (68778, 7)
Weather shape: (3182, 6)


Check duplicates and aggregate inverter readings

In [9]:
duplicate_records = gen.duplicated(["DATE_TIME", "SOURCE_KEY"]).sum()
print("Duplicate inverter-timestamp records:", duplicate_records)

if duplicate_records:
    raise ValueError("Duplicate records found for an inverter and timestamp.")

if wth["DATE_TIME"].duplicated().any():
    raise ValueError("Weather data contains duplicate timestamps.")

N_INVERTERS = 22

gen_plant = (
    gen.groupby("DATE_TIME")
    .agg(
        DC_POWER=("DC_POWER", "sum"),
        AC_MEAN=("AC_POWER", "mean"),
        DAILY_YIELD=("DAILY_YIELD", "sum"),
        TOTAL_YIELD=("TOTAL_YIELD", "sum"),
        N_INV=("SOURCE_KEY", "nunique"),
    )
    .reset_index()
)

# Estimate plant total from the mean of the inverters reporting at each time.
gen_plant["AC_POWER"] = gen_plant["AC_MEAN"] * N_INVERTERS

incomplete = gen_plant["N_INV"] < N_INVERTERS
daytime = gen_plant["DATE_TIME"].dt.hour.between(6, 17)

print(
    f"Timestamps with fewer than {N_INVERTERS} inverters:",
    int(incomplete.sum()),
    "of",
    len(gen_plant),
)
print("Of those, daytime timestamps:", int((incomplete & daytime).sum()))

gen_plant = gen_plant.drop(columns=["AC_MEAN", "N_INV"])

Duplicate inverter-timestamp records: 0
Timestamps with fewer than 22 inverters: 101 of 3158
Of those, daytime timestamps: 10


Merge the generation and weather data

In [10]:
weather_columns = [
    "DATE_TIME",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "IRRADIATION",
]

wth_clean = wth[weather_columns].copy()

merged = (
    pd.merge(
        gen_plant,
        wth_clean,
        on="DATE_TIME",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("DATE_TIME")
    .reset_index(drop=True)
)

print("Merged shape:", merged.shape)
print("Missing values:", int(merged.isna().sum().sum()))
print("Date range:", merged["DATE_TIME"].min(), "to", merged["DATE_TIME"].max())

if merged.isna().any().any():
    raise ValueError("Missing values found after merging.")

Merged shape: (3157, 8)
Missing values: 0
Date range: 2020-05-15 00:00:00 to 2020-06-17 23:45:00


##Prepare the modeling data and define forecast horizons
The cumulative yield columns and DC power are excluded from the model features. The horizons are measured in 15-minute steps: 1 step is 15 minutes, 2 is 30 minutes, 3 is 45 minutes and 4 is 60 minutes.

In [11]:
STEP = pd.Timedelta(minutes=15)

HORIZONS = {
    1: "15min",
    2: "30min",
    3: "45min",
    4: "60min",
}

N_LAGS = 3

base = (
    merged.drop(columns=["DAILY_YIELD", "TOTAL_YIELD", "DC_POWER"])
    .sort_values("DATE_TIME")
    .reset_index(drop=True)
)

if not base["DATE_TIME"].is_unique:
    raise ValueError("Duplicate timestamps found after aggregation.")

ac_by_time = base.set_index("DATE_TIME")["AC_POWER"]

print("Modeling data shape:", base.shape)
print("Modeling columns:", base.columns.tolist())

Modeling data shape: (3157, 5)
Modeling columns: ['DATE_TIME', 'AC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']


Build one dataset for each horizon

In [12]:
def build_dataset(base, h, n_lags=3):
    """
    Build features and target for a horizon of h 15-minute steps.
    """
    df = base.copy()
    times = df["DATE_TIME"]

    # Past power values at exact timestamps.
    for k in range(1, n_lags + 1):
        lag_times = times - k * STEP
        df[f"AC_POWER_lag{k}"] = ac_by_time.reindex(lag_times).to_numpy()

    # Future target at the exact target timestamp.
    df["TARGET_TIME"] = times + h * STEP
    df["TARGET"] = ac_by_time.reindex(df["TARGET_TIME"]).to_numpy()

    # Time-of-day features, known at forecast time.
    minutes = times.dt.hour * 60 + times.dt.minute
    df["hour_sin"] = np.sin(2 * np.pi * minutes / 1440)
    df["hour_cos"] = np.cos(2 * np.pi * minutes / 1440)

    required_columns = [
        "TARGET",
        *[f"AC_POWER_lag{k}" for k in range(1, n_lags + 1)],
    ]

    rows_before = len(df)
    df = df.dropna(subset=required_columns).reset_index(drop=True)
    rows_dropped = rows_before - len(df)

    return df, rows_dropped


datasets = {}

for h, label in HORIZONS.items():
    dataset, dropped = build_dataset(base, h, N_LAGS)
    datasets[h] = dataset

    print(
        f"{label}: {len(dataset)} rows kept, "
        f"{dropped} dropped, {dataset.shape[1]} columns"
    )

print("\nColumns:")
print(datasets[1].columns.tolist())

15min: 3113 rows kept, 44 dropped, 12 columns
30min: 3105 rows kept, 52 dropped, 12 columns
45min: 3098 rows kept, 59 dropped, 12 columns
60min: 3092 rows kept, 65 dropped, 12 columns

Columns:
['DATE_TIME', 'AC_POWER', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION', 'AC_POWER_lag1', 'AC_POWER_lag2', 'AC_POWER_lag3', 'TARGET_TIME', 'TARGET', 'hour_sin', 'hour_cos']


Verify the target and every lag by timestamp

In [13]:
lookup = dict(zip(base["DATE_TIME"], base["AC_POWER"]))

for h, dataset in datasets.items():
    expected_target = dataset["DATE_TIME"].map(
        lambda t: lookup.get(t + h * STEP, np.nan)
    )

    target_matches = np.isclose(
        expected_target.to_numpy(dtype=float),
        dataset["TARGET"].to_numpy(dtype=float),
        equal_nan=False,
    )

    print(f"\n{HORIZONS[h]} target mismatches:", int((~target_matches).sum()))

    for k in range(1, N_LAGS + 1):
        expected_lag = dataset["DATE_TIME"].map(
            lambda t: lookup.get(t - k * STEP, np.nan)
        )

        lag_matches = np.isclose(
            expected_lag.to_numpy(dtype=float),
            dataset[f"AC_POWER_lag{k}"].to_numpy(dtype=float),
            equal_nan=False,
        )

        print(
            f"{HORIZONS[h]} lag {k} mismatches:",
            int((~lag_matches).sum()),
        )

    correct_target_time = (
        dataset["TARGET_TIME"] - dataset["DATE_TIME"] == h * STEP
    ).all()

    print(f"{HORIZONS[h]} target timestamps correct:", correct_target_time)


15min target mismatches: 0
15min lag 1 mismatches: 0
15min lag 2 mismatches: 0
15min lag 3 mismatches: 0
15min target timestamps correct: True

30min target mismatches: 0
30min lag 1 mismatches: 0
30min lag 2 mismatches: 0
30min lag 3 mismatches: 0
30min target timestamps correct: True

45min target mismatches: 0
45min lag 1 mismatches: 0
45min lag 2 mismatches: 0
45min lag 3 mismatches: 0
45min target timestamps correct: True

60min target mismatches: 0
60min lag 1 mismatches: 0
60min lag 2 mismatches: 0
60min lag 3 mismatches: 0
60min target timestamps correct: True


In [14]:
for h, dataset in datasets.items():
    output_path = folder / f"df_clean_h{h}.csv"
    dataset.to_csv(output_path, index=False)
    print("Saved:", output_path, "Shape:", dataset.shape)

Saved: /content/drive/MyDrive/Project2_SolarPV/df_clean_h1.csv Shape: (3113, 12)
Saved: /content/drive/MyDrive/Project2_SolarPV/df_clean_h2.csv Shape: (3105, 12)
Saved: /content/drive/MyDrive/Project2_SolarPV/df_clean_h3.csv Shape: (3098, 12)
Saved: /content/drive/MyDrive/Project2_SolarPV/df_clean_h4.csv Shape: (3092, 12)
